The data in the file ej5tarea5.sav correspond to 700 clients who were either denied or not denied credit (in the dataset, this is represented by the variable default, which takes the value 1 if the credit was denied and 0 if it was not).
At the same time, various socioeconomic variables were measured for each client, some of which influenced that decision.

The purpose of a new analysis is to determine a decision rule that allows us to establish, for new clients, whether credit should be granted or not.

 x1= Age in years; 
 x2= Level of education (1 = Did not complete high school, 2 = High school, 3 = Some college, 4 = College, 5 = Post undergraduate degree); 
 x3= Years with current employer; 
 x4= Years at current address; 
 x5 = Household income in thousands; 
 x6= Debt-to-income ratio (×100); 
 x7= Credit card debt in thousands; 
 x8= Other debt in thousands; 
 x9 = Previously defaulted (Historial de incumplimiento), (1 if the credit was denied and 0 if it was not)

 Solution:
 1) Classification problem
 2) Use logistic regression
 3) Analyze for unbalanced data (it would be less than 20%)
 4) Build the structural model (significant variables)
 5) Try different tresholds. 

 ************  Note: it is important to consider scaling the data

"Data-splitting and model evaluation workflow"

STEPS:

Dataset split (n=700)

Step 1: Split into 4 equal parts (each = 25%)
Total data (700)
│
├── Part 1 → 175 samples
├── Part 2 → 175 samples
├── Part 3 → 175 samples
├── Part 4 → 175 samples

Step 2: Build the 2:1:1 structure
 Training set (2 parts = 50%) → teaches the model

Part 1 + Part 2 = 350 samples = Training dataset
→  To train logistic regression model and evaluate the significance of the variables
Validation set (1 part = 25%) 

Part 3 = 175 samples = Validation set
→ To choose the best threshold
 Test set (1 part = 25%) → checks generalization
 
Part 4 = 175 samples = Test set
→ For final evaluation


# **************************************************                    Libraries and reading the data               **************************************************#

In [41]:
#Libraries
%matplotlib notebook
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# install the library
import sys
!{sys.executable} -m pip install pyreadstat

import pandas as pd
import numpy as np

pd.options.display.float_format = '{:.2f}'.format

bank = pd.read_spss('C:/Users/paola/OneDrive/Documentos/LUMIA_Data_Scientist/ej5tarea5.sav')

print(bank.head())

feature_names_bank = ['age', 'ed', 'employ', 'address', 'income', 'debtinc','creddebt', 'othdebt']
X_bank  = bank [feature_names_bank ]

# the only dummy variable is education
# Convert 'ed' into dummy variables
X_bank = pd.get_dummies(X_bank, columns=['ed'], drop_first=True)

print(X_bank.head())

#Variable de output es cuantitativa
y_bank  = bank ['default']
print(y_bank .head())

    age                            ed  employ  address  income  debtinc  \
0 41.00                  Some college   17.00    12.00  176.00     9.30   
1 27.00  Did not complete high school   10.00     6.00   31.00    17.30   
2 40.00  Did not complete high school   15.00    14.00   55.00     5.50   
3 41.00  Did not complete high school   15.00    14.00  120.00     2.90   
4 24.00            High school degree    2.00     0.00   28.00    17.30   

   creddebt  othdebt default  ZRE_1  
0     11.36     5.01     Yes   0.49  
1      1.36     4.00      No  -0.50  
2      0.86     2.17      No  -0.10  
3      2.66     0.82      No  -0.15  
4      1.79     3.06     Yes   0.53  
    age  employ  address  income  debtinc  creddebt  othdebt  \
0 41.00   17.00    12.00  176.00     9.30     11.36     5.01   
1 27.00   10.00     6.00   31.00    17.30      1.36     4.00   
2 40.00   15.00    14.00   55.00     5.50      0.86     2.17   
3 41.00   15.00    14.00  120.00     2.90      2.66     0.82   
4


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
# Output variable with labels
y_bank = bank['default']
# Associated values per label
target_names_bank = ['No', 'Yes']
target_names_bank

['No', 'Yes']

In [33]:
# Split into training and test samples, by default 75% training, with seed set to zero
X_train, X_test, y_train, y_test = train_test_split(X_bank, y_bank, random_state=0)

#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#

Exploring the confusion matrix / Dummy Classifier and Logistic regression, before the "Data-splitting and model evaluation workflow":

#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#

# *********************************************************             CONFUSION MATRIX                ********************************************************** #

Which confusion matrix should be used?


First type is: Dummy classifier → baseline
The real model (e.g., logistic regression) → actual performance

Comparison:

If the model is not better than the dummy → ❌ model is useless
If it improves FN/FP → ✅ model is learning something

Important notes about confusion matrix

The confusion matrix isn’t something you “choose” between DummyClassifier or Bernoulli classifier. It’s an evaluation output you compute after making predictions with any model, including logistic regression.

A confusion matrix is tied to:

your true labels (y_true)
your model’s predicted labels (y_pred)

It doesn’t depend on the type of classifier—you can compute it for:

Logistic regression
DummyClassifier (baseline)
Bernoulli Naive Bayes
Any classifier

Bottom line
You don’t choose a confusion matrix type
You choose a model, then compute its confusion matrix
DummyClassifier is just a baseline, not something to replace logistic regression with unless you're benchmarking

In [34]:

# Fitting DummyClassifier

from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# Split data (if not already done)
X_train, X_test, y_train, y_test = train_test_split(
    X_bank, y_bank, test_size=0.3, random_state=42
)

# Dummy classifier (majority class)
dummy_majority = DummyClassifier(strategy='most_frequent')
dummy_majority.fit(X_train, y_train)

# Predictions
y_majority_predicted = dummy_majority.predict(X_test)

# Confusion matrix
confusion = confusion_matrix(y_test, y_majority_predicted)


print("Most frequent class (dummy classifier)\n", confusion)

Most frequent class (dummy classifier)
 [[161   0]
 [ 49   0]]


# ..............................                   Interpretation of the confusion matrix with Dummy Classifier                       .............................. #

TN = 161 → correctly predicted no default (0) ✅
FP = 0 → predicted default when it wasn’t ❌ (none)
FN = 49 → predicted no default, but actually default ❗
TP = 0 → correctly predicted default (1) ❌
This dummy classifier is using “most frequent class”, so:

It predicts ALL observations as 0 (no default)

📉 Model behavior
It is perfect at predicting non-defaulters
It completely fails to detect defaulters
Key issue:
49 people actually defaulted, and the model missed ALL of them ❌

In a credit context:

❗ FN = 49 (VERY BAD)
→ You would approve 49 risky clients who will default
✅ TN = 161
→ You correctly approve safe clients
❌ TP = 0
→ You detect zero risky clients

✅ Final takeaway
This dummy model is your baseline
Your logistic regression model should:
Increase TP (detect defaulters)
Reduce FN

The model needs improvement (e.g., class imbalance handling)

# ***********************************************************             2) LOGISTIC REGRESSION               ******************************************************** #

 Compare the confusion matrix of the dummy classifier with the one of logistic regression (we could also compare against the Bernoulli confusion matrix)

In [35]:
# Library for logistic regression, with dummy classifier confusion matrix
from sklearn.linear_model import LogisticRegression
# Model predictions
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
lr_predicted = lr.predict(X_test)
# confusion matrix
confusion = confusion_matrix(y_test, lr_predicted)
# Similar results
print('Logistic regression classifier (default settings)\n', confusion)

Logistic regression classifier (default settings)
 [[149  12]
 [ 18  31]]


# ....................................          Conclusion for the confusion matrix with logistic regression classifier                ......................................... #


The earlier confusion matrix (Dummy classifier) showed poor performance.
Even if some variables are “significant”, this new confusion matrix indicates that the model may still be weak for prediction due to class imbalance. Imbalanced data issues need to be addressed.

In [28]:
bank.shape

(700, 10)

Re-encoding a target variable into a binary (0/1) format, before training something like Logistic Regression. There are many more 0s than 1s (or vice versa)

In [36]:
# Creating a dataset with imbalanced binary classes:  
# Negative class (0) is 'not digit 1' 
# Positive class (1) is 'digit 1'
# X_bank, y_bank
# Takes the original labels in y_bank (which are strings like "Yes" and "No")
# Converts them into numeric values:
# "Yes" → 1 *credit denied
# "No" → 0
y_binary_imbalanced = y_bank.map({'Yes': 1, 'No': 0})
# Forcing everything that is not 1 to be 0
y_binary_imbalanced[y_binary_imbalanced != 1] = 0 

# Check results
print('Original labels:\t', y_bank.iloc[1:30])
print('New binary labels:\t', y_binary_imbalanced.iloc[1:30])


Original labels:	 1      No
2      No
3      No
4     Yes
5      No
6      No
7      No
8     Yes
9      No
10     No
11     No
12     No
13     No
14     No
15    Yes
16    Yes
17     No
18     No
19     No
20     No
21     No
22     No
23     No
24    Yes
25     No
26     No
27     No
28     No
29     No
Name: default, dtype: category
Categories (2, str): ['No', 'Yes']
New binary labels:	 1     0
2     0
3     0
4     1
5     0
6     0
7     0
8     1
9     0
10    0
11    0
12    0
13    0
14    0
15    1
16    1
17    0
18    0
19    0
20    0
21    0
22    0
23    0
24    1
25    0
26    0
27    0
28    0
29    0
Name: default, dtype: category
Categories (2, int64): [0, 1]


In [37]:
np.bincount(y_binary_imbalanced)    

array([517, 183])

# ****************************************************************         3)  IMBALANCED DATA             **************************************************************** #  

Negative class (0) is the most frequent class (517 are cero (credit not denied) y 183 (1, credit denied))

IMBALANCED DATA

It is when is 20% in one class
We have:
517 (73.85%) vs 183 (26.14%)

We proceed as if we have imbalanced data

What you should do
1. Stop relying on accuracy

Use better metrics derived from the confusion matrix:

Precision
Recall (very important for imbalance)
F1-score (We use this one, the higher the better)
ROC-AUC

Focus especially on:

Recall for class 1 (can you detect the rare cases?)

2. Use class weights (simple and powerful)

For Logistic Regression:

model = LogisticRegression(class_weight='balanced')

This:

Penalizes mistakes on the minority class more
Often the first thing you should try

3. Change the decision threshold

Default is:

threshold = 0.5

But for imbalance, try lowering it:

y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.3).astype(int)

Lower threshold → more positives detected → higher recall

4. Resample the data
Option A: Oversampling (add more minority cases)
Duplicate or synthesize minority samples
Common method: SMOTE
Option B: Undersampling (reduce majority class)
Remove some majority samples

Trade-off:

Oversampling → risk of overfitting
Undersampling → lose information

5. Compare against a baseline

Use DummyClassifier:

from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")

If your model isn’t better than this → something’s wrong.

6. Look at the confusion matrix properly

Instead of:

TN FP
FN TP

Focus on:

FN (false negatives) → missed important cases
TP (true positives) → what you care about


Then we will:
Train logistic regression
Add class_weight='balanced'
Evaluate with confusion matrix + recall/F1
Tune threshold
Try resampling if needed
Compare vs DummyClassifier and see if it is better

First solution:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')

In [38]:
# with balanced data 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_bank, y_binary_imbalanced, random_state=0
)

# Add class_weight='balanced' here 👇
lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)

lr_predicted = lr.predict(X_test)

# Confusion matrix
confusion = confusion_matrix(y_test, lr_predicted)

print('Logistic regression classifier (balanced)\n', confusion)



Logistic regression classifier (balanced)
 [[96 35]
 [15 29]]


# .......................    Interpretation of the confusion matrix with lr = LogisticRegression(max_iter=1000, class_weight='balanced') ......................... #  

TN = 96 → correctly predicted negatives
FP = 35 → predicted 1 but actually 0
FN = 15 → missed positives
TP = 29 → correctly predicted positives
📈 Key metrics
Accuracy
(96 + 29) / 175 = 125 / 175 ≈ 0.71

➡️ 71% accuracy (lower than before)

Recall (for class 1) ⭐
Recall = 29 / (29 + 15) ≈ 0.66

➡️ 66% of positives detected

✅ Much better than before (~39%)

Precision
Precision = 29 / (29 + 35) ≈ 0.45

➡️ Only 45% of predicted positives are correct

⚠️ More false alarms

🧠 What changed compared to your previous matrix

Before (unbalanced model):

TP = 17
FN = 27
Recall ≈ 39%

Now (balanced model):

TP = 29 ✅ (big improvement)
FN = 15 ✅ (much fewer missed cases)
Recall ≈ 66% ✅

Bottom line
Old model: safe but blind (missed many positives)
New model: more sensitive (detects more positives, but noisier)

👉 This is exactly what class_weight='balanced' is supposed to do.

👍 Final insight

You improved the most important metric for imbalanced data:
➡️ Recall (39% → 66%)

That’s a meaningful upgrade—even though accuracy dropped.

The next step is choosing the best threshold to balance precision vs recall.

Even the model with all significant value can perfomr not so good because of the threshold.


# ** 4) Build the structural model (significant variables), printing the p values and coefficients. We will take as the optimum model as the one with significant variables. ** #


#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#
                                                         "Data-splitting and model evaluation workflow":

#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#

# .................................................        Step 1: Split into 4 equal parts (each = 25%)       ................................................#


# the complete data is:  X_bank, y_binary_imbalanced

Step 1: Split into 4 equal parts (each = 25%)
Total data (700)
│
├── Part 1 → 175 samples
├── Part 2 → 175 samples
├── Part 3 → 175 samples
├── Part 4 → 175 samples

Step 2: Build the 2:1:1 structure
 Training set (2 parts = 50%) → teaches the model

Part 1 + Part 2 = 350 samples = Training dataset
→  To train logistic regression model and evaluate the significance of the variables
Validation set (1 part = 25%) 

In [59]:
import numpy as np
from sklearn.utils import shuffle

# Shuffle
X_bank_shuffled, y_shuffled = shuffle(X_bank, y_binary_imbalanced, random_state=0)

# Split
X_parts = np.array_split(X_bank_shuffled, 4)
y_parts = np.array_split(y_shuffled, 4)

# Assign
X_part1, X_part2, X_part3, X_part4 = X_parts
y_part1, y_part2, y_part3, y_part4 = y_parts

# Combine Part 1 + Part 2
X_combined = np.concatenate([X_part1, X_part2], axis=0)
y_combined = np.concatenate([y_part1, y_part2], axis=0)

print(X_combined.shape, y_combined.shape)

(350, 11) (350,)


# **************************  Fitting the regression model to get the significant variables according to the p value, using the train dataset   *************************  #

You can get coefficients and p-values only if you refit the model using statsmodels, because:

scikit-learn → gives coefficients only (no p-values)
statsmodels → gives coefficients + p-values (for inference)

The threshold (even the optimum) does NOT affect coefficients or p-values.

Coefficients come from model fitting
Threshold only affects final predictions / confusion matrix

So:

You compute coefficients & p-values once
Then you can apply any threshold (0.5, 0.38, etc.) afterward

In [ ]:
# insert the variable names
X_combined.columns = X_bank.columns

In [ ]:
# Use GLM instead of Logit if you want weights
# Ensure numeric data (no objects/strings)
# fit the model with statsmodels

# the datasets are: X_combined, y_combined 

import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# set the variable names :

# Convert to DataFrame explicitly
X_combined = pd.DataFrame(X_combined)
y_combined  = pd.Series(y_combined )

# Convert ALL columns to numeric
X_combined = X_combined.apply(pd.to_numeric, errors='coerce')
y_combined  = pd.to_numeric(y_combined , errors='coerce')

# Combine and drop NaNs
data = pd.concat([X_combined, y_combined .rename("target")], axis=1)
data = data.dropna()

X_combined = data.drop(columns=["target"])
y_combined  = data["target"]

# 🔴 Force float dtype (this fixes your error)
X_combined = X_combined.astype(float)
y_combined  = y_combined .astype(float)

# Compute weights AFTER cleaning
sample_weights = compute_sample_weight(class_weight='balanced', y=y_combined )

# Add intercept
X_combined_sm = sm.add_constant(X_combined, has_constant='add')

# Final safety check (important)
print(X_combined_sm.dtypes.unique())  # should be only float64
print(y_combined .dtype)

# Use GLM with weights passed correctly
model = sm.GLM(
    y_combined ,
    X_combined_sm,
    family=sm.families.Binomial(),
    freq_weights=sample_weights
)

result = model.fit()

print(result.summary())



[dtype('float64')]
float64
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  350
Model:                            GLM   Df Residuals:                      338
Model Family:                Binomial   Df Model:                           11
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -163.80
Date:                Wed, 22 Apr 2026   Deviance:                       327.60
Time:                        09:20:45   Pearson chi2:                     325.
No. Iterations:                    20   Pseudo R-squ. (CS):             0.3626
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

# ************************************************************************   Conclusion: ************************************************************************   #
The education variables are the ones with the highest p value, but since they are dummy variables (and non of them are significant), we can drop all of them.

In [61]:
print(X_combined.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')


In [62]:
# drop the education variables
X_combined  = X_combined.drop(columns=[
    'ed_Did not complete high school',
    'ed_High school degree',
    'ed_Post-undergraduate degree',
    'ed_Some college'
])
print(X_combined.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt'], dtype='str')


Fitting the model (train dataset) without the education variables

In [64]:
# Use GLM instead of Logit if you want weights
# Ensure numeric data (no objects/strings)
# fit the model with statsmodels

# the datasets are: X_combined, y_combined 

import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# Convert to DataFrame explicitly
X_combined = pd.DataFrame(X_combined)
y_combined  = pd.Series(y_combined )

# Convert ALL columns to numeric
X_combined = X_combined.apply(pd.to_numeric, errors='coerce')
y_combined  = pd.to_numeric(y_combined , errors='coerce')

# Combine and drop NaNs
data = pd.concat([X_combined, y_combined .rename("target")], axis=1)
data = data.dropna()

X_combined = data.drop(columns=["target"])
y_combined  = data["target"]

# 🔴 Force float dtype (this fixes your error)
X_combined = X_combined.astype(float)
y_combined  = y_combined .astype(float)

# Compute weights AFTER cleaning
sample_weights = compute_sample_weight(class_weight='balanced', y=y_combined )

# Add intercept
X_combined_sm = sm.add_constant(X_combined, has_constant='add')

# Final safety check (important)
print(X_combined_sm.dtypes.unique())  # should be only float64
print(y_combined .dtype)

# Use GLM with weights passed correctly
model = sm.GLM(
    y_combined ,
    X_combined_sm,
    family=sm.families.Binomial(),
    freq_weights=sample_weights
)

result = model.fit()

print(result.summary())

[dtype('float64')]
float64
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  350
Model:                            GLM   Df Residuals:                      342
Model Family:                Binomial   Df Model:                            7
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -165.17
Date:                Wed, 22 Apr 2026   Deviance:                       330.34
Time:                        09:23:28   Pearson chi2:                     325.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3576
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.6041    

# ************************************************************************   Conclusion: ************************************************************************   #
We drop the variable othdebt (p value = 0.286)

# *************************                            Fitting the logistic regression model without education and othdebt                      *********************   #

In [66]:
print(X_combined.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt'], dtype='str')


In [68]:
# drop the education variables
X_combined  = X_combined.drop(columns=[
    'othdebt'
])
print(X_combined.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt'], dtype='str')


In [70]:
# Use GLM instead of Logit if you want weights
# Ensure numeric data (no objects/strings)
# fit the model with statsmodels

# the datasets are: X_combined, y_combined 

import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# set the variable names :

# Convert to DataFrame explicitly
X_combined = pd.DataFrame(X_combined)
y_combined  = pd.Series(y_combined )

# Convert ALL columns to numeric
X_combined = X_combined.apply(pd.to_numeric, errors='coerce')
y_combined  = pd.to_numeric(y_combined , errors='coerce')

# Combine and drop NaNs
data = pd.concat([X_combined, y_combined .rename("target")], axis=1)
data = data.dropna()

X_combined = data.drop(columns=["target"])
y_combined  = data["target"]

# 🔴 Force float dtype (this fixes your error)
X_combined = X_combined.astype(float)
y_combined  = y_combined .astype(float)

# Compute weights AFTER cleaning
sample_weights = compute_sample_weight(class_weight='balanced', y=y_combined )

# Add intercept
X_combined_sm = sm.add_constant(X_combined, has_constant='add')

# Final safety check (important)
print(X_combined_sm.dtypes.unique())  # should be only float64
print(y_combined .dtype)

# Use GLM with weights passed correctly
model = sm.GLM(
    y_combined ,
    X_combined_sm,
    family=sm.families.Binomial(),
    freq_weights=sample_weights
)

result = model.fit()

print(result.summary())

[dtype('float64')]
float64
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  350
Model:                            GLM   Df Residuals:                      343
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -165.76
Date:                Wed, 22 Apr 2026   Deviance:                       331.51
Time:                        09:30:23   Pearson chi2:                     327.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3554
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.9399    

# ************************************************************************   Conclusion: ************************************************************************   #
We drop the variable income (p value = 0.386)

# *************************                            Fitting the logistic regression model without education, othdebt and income                     *********************   #

In [71]:
# drop the education variables
X_combined  = X_combined.drop(columns=[
    'income'
])
print(X_combined.columns)

Index(['age', 'employ', 'address', 'debtinc', 'creddebt'], dtype='str')


In [159]:
# Use GLM instead of Logit if you want weights
# Ensure numeric data (no objects/strings)
# fit the model with statsmodels

# the datasets are: X_combined, y_combined 

import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# set the variable names :

# Convert to DataFrame explicitly
X_combined = pd.DataFrame(X_combined)
y_combined  = pd.Series(y_combined )

# Convert ALL columns to numeric
X_combined = X_combined.apply(pd.to_numeric, errors='coerce')
y_combined  = pd.to_numeric(y_combined , errors='coerce')

# Combine and drop NaNs
data = pd.concat([X_combined, y_combined .rename("target")], axis=1)
data = data.dropna()

X_combined = data.drop(columns=["target"])
y_combined  = data["target"]

# 🔴 Force float dtype (this fixes your error)
X_combined = X_combined.astype(float)
y_combined  = y_combined .astype(float)

# Compute weights AFTER cleaning
sample_weights = compute_sample_weight(class_weight='balanced', y=y_combined )

# Add intercept
X_combined_sm = sm.add_constant(X_combined, has_constant='add')

# Final safety check (important)
print(X_combined_sm.dtypes.unique())  # should be only float64
print(y_combined .dtype)

# Use GLM with weights passed correctly
model = sm.GLM(
    y_combined ,
    X_combined_sm,
    family=sm.families.Binomial(),
    freq_weights=sample_weights
)

result = model.fit()

print(result.summary())

[dtype('float64')]
float64
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  350
Model:                            GLM   Df Residuals:                      344
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -166.15
Date:                Thu, 23 Apr 2026   Deviance:                       332.30
Time:                        09:41:14   Pearson chi2:                     329.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3540
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.0330    

# ************************************************************************   Conclusion: ************************************************************************   #
The significant variables are age, employ, address, debtinc, creddebt

#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#
# ***                                                                         Next Steps:                                                                               ***   #
Use the Validation set:
Fit the model with the significant variables  age, employ, address, debtinc, creddebt.
The model is still using the default threshold = 0.5, which is why we are getting that precision/recall trade-off. To choose the best threshold, we test different values in the validation set.
#*****************************************************************************************************************************************************************************#
#*****************************************************************************************************************************************************************************#

In [129]:
# build the datasets for x and y as validation sets
import pandas as pd

X_validation_set = pd.DataFrame(X_part3, columns=X_bank.columns)
Y_validation_set = pd.DataFrame(y_part3)
print(X_validation_set.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')


In [130]:
print(X_bank.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')


In [131]:
# X_validation_set  must have only the significant variables  age, employ, address, debtinc, creddebt

print(X_validation_set.columns)

X_validation_set  =  X_validation_set.drop(columns=['income','othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'
])
print(X_validation_set.columns)



Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')
Index(['age', 'employ', 'address', 'debtinc', 'creddebt'], dtype='str')


In [158]:
# the datasets are:
# X_validation_set 
# Y_validation_set

# fitting the model with the significant variables 
# with balanced data 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# we do not divide in train and set, the validation set. We took it as one
# X_train, X_test, y_train, y_test = train_test_split(
#    X_validation_set , Y_validation_set, random_state=0
# )

# Add class_weight='balanced' here 👇
lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_validation_set , Y_validation_set)

lr_predicted = lr.predict(X_validation_set)

# Confusion matrix
confusion = confusion_matrix(Y_validation_set,lr_predicted)

print('Logistic regression classifier (balanced)\n', confusion)


Logistic regression classifier (balanced)
 [[99 25]
 [ 8 43]]


c:\Users\paola\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


# ***********************            Interpretation of the confusion matrix with significant variables and threshold of 0.5    ****************************#  

Key metrics (from this matrix)

Let’s compute the intuition:

Accuracy = (99 + 43) / total = 142 / 175 ≈ 0.81
Precision (class 1) = 43 / (43 + 25) ≈ 0.63
Recall (class 1) = 43 / (43 + 8) ≈ 0.84
F1 score ≈ 0.72

🧠 Interpretation
✔️ Strengths
High recall (0.84) → the model catches most positives
Good number of true positives (43)
Few missed positives (only 8)

👉 This is good if missing positives is costly (e.g., credit risk, fraud)

⚠️ Weakness
False positives are relatively high (25)
Precision is moderate (0.63)

👉 The model sometimes flags negatives as positives
Practical meaning

If this were a credit denial model:

✅ 43 risky clients correctly flagged
⚠️ 25 safe clients incorrectly flagged (extra review needed)
❗ 8 risky clients missed (important but not huge)


Next steps:
find the optimal threshold to reduce those 25 false positives

# ************************************************             Finding the optimal threshold                 ***************************************** #

# the datasets are: X_validation_set, Y_validation_set

In [133]:
# Step 1: Get probabilities (instead of hard predictions)
lr_predicted = lr.predict(X_validation_set)
y_prob = lr.predict_proba(X_validation_set)[:, 1]

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.linspace(0.1, 0.9, 9)

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    
    # precision = precision_score(y_test, y_pred)
    # recall = recall_score(y_test, y_pred)
    # f1 = f1_score(y_test, y_pred)

    precision = precision_score(Y_validation_set, y_pred)
    recall = recall_score(Y_validation_set, y_pred)
    f1 = f1_score(Y_validation_set, y_pred)
    
    print(f"Threshold: {t:.2f} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f}")

Threshold: 0.10 | Precision: 0.40 | Recall: 0.98 | F1: 0.57
Threshold: 0.20 | Precision: 0.47 | Recall: 0.92 | F1: 0.62
Threshold: 0.30 | Precision: 0.53 | Recall: 0.92 | F1: 0.68
Threshold: 0.40 | Precision: 0.57 | Recall: 0.88 | F1: 0.69
Threshold: 0.50 | Precision: 0.63 | Recall: 0.84 | F1: 0.72
Threshold: 0.60 | Precision: 0.67 | Recall: 0.78 | F1: 0.72
Threshold: 0.70 | Precision: 0.80 | Recall: 0.69 | F1: 0.74
Threshold: 0.80 | Precision: 0.85 | Recall: 0.55 | F1: 0.67
Threshold: 0.90 | Precision: 0.94 | Recall: 0.31 | F1: 0.47


# ****************************   CHOOSE THE BEST THRESHOLD FROM THE LIST, THE ONE THAT MAXIMIZES F1 (WHERE PRECISION IS SIMILAR TO RECALL)  ************************************#

Choosing the “best” threshold depends on your goal:

Option A: Balanced performance (recommended)

Pick the threshold with highest F1-score

Option B: Prioritize recall (catch more positives)

Pick a lower threshold (e.g., 0.3–0.4)

Option C: Prioritize precision (fewer false alarms)

Pick a higher threshold (e.g., 0.6–0.7)

Better method: Precision–Recall curve

Bottom line
predict() locks you at 0.5 threshold
Use predict_proba() to take control
Choose threshold based on:
F1-score (best balance) ✅, the threshold where F1-score is highest
or your business goal (recall vs precision)

In [134]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(Y_validation_set, y_prob)

for i in range(len(thresholds)):
    precision = precisions[i]
    recall = recalls[i]
    
    # avoid division by zero
    if (precision + recall) == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    
    print(f"Threshold: {thresholds[i]:.2f}, "
          f"Precision: {precision:.2f}, "
          f"Recall: {recall:.2f}, "
          f"F1: {f1:.2f}")

Threshold: 0.00, Precision: 0.29, Recall: 1.00, F1: 0.45
Threshold: 0.00, Precision: 0.29, Recall: 1.00, F1: 0.45
Threshold: 0.00, Precision: 0.29, Recall: 1.00, F1: 0.46
Threshold: 0.00, Precision: 0.30, Recall: 1.00, F1: 0.46
Threshold: 0.00, Precision: 0.30, Recall: 1.00, F1: 0.46
Threshold: 0.01, Precision: 0.30, Recall: 1.00, F1: 0.46
Threshold: 0.01, Precision: 0.30, Recall: 1.00, F1: 0.46
Threshold: 0.01, Precision: 0.30, Recall: 1.00, F1: 0.47
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.47
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.47
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.47
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.47
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.48
Threshold: 0.01, Precision: 0.31, Recall: 1.00, F1: 0.48
Threshold: 0.01, Precision: 0.32, Recall: 1.00, F1: 0.48
Threshold: 0.01, Precision: 0.32, Recall: 1.00, F1: 0.48
Threshold: 0.01, Precision: 0.32, Recall: 1.00, F1: 0.49
Threshold: 0.01, Precision: 0.3

 # *************************************************                       Conclusion                         ************************************************* #

Then choose:
where F1 is highest  

Threshold:	0.69,	Precision:	0.79,	Recall:	0.73,	F1:	0.76
Threshold:	0.69,	Precision:	0.77,	Recall:	0.73,	F1:	0.75
Threshold:	0.70,	Precision:	0.80,	Recall:	0.71,	F1:	0.75
Threshold:	0.61,	Precision:	0.70,	Recall:	0.78,	F1:	0.74
Threshold:	0.69,	Precision:	0.76,	Recall:	0.73,	F1:	0.74
Threshold:	0.69,	Precision:	0.78,	Recall:	0.71,	F1:	0.74


Threshold of 0.69, is the one that gives the highest F1, with precision similar to recall (less distance between them) :
Threshold:	0.69,	Precision:	0.79,	Recall:	0.73,	F1:	0.76, but the precision is not so good. 

Then we decide for a higher precision (that is similar to recall)

Threshold:	0.69,	Precision:	0.79		Recall:	0.73	F1:	0.76		0.06	0.06
Threshold:	0.75,	Precision:	0.79		Recall:	0.61	F1:	0.69		0.18	0.18
Threshold:	0.75,	Precision:	0.79		Recall:	0.59	F1:	0.67		0.2	0.2
Threshold:	0.69,	Precision:	0.78		Recall:	0.71	F1:	0.74		0.07	0.07
Threshold:	0.69,	Precision:	0.77		Recall:	0.73	F1:	0.75		0.04	0.04
Threshold:	0.69,	Precision:	0.76		Recall:	0.73	F1:	0.74		0.03	0.03
Threshold:	0.69,	Precision:	0.74		Recall:	0.73	F1:	0.73		0.01	0.01
Threshold:	0.67,	Precision:	0.73		Recall:	0.73	F1:	0.73		0	0

The best is again threshold = 0.69


What will happen when you change threshold
T hreshold	        Effect
Lower (0.3)	↑ Recall, ↓ Precision
Higher (0.7)	↑ Precision, ↓ Recall

# ******************************************************** APPLY THE THRESHOLD OF 0.69 IN THE VALIDATION SET TO GET THE METRICS *********************************************** #

In [135]:
y_pred_final = (y_prob >= 0.69).astype(int)
confusion = confusion_matrix(Y_validation_set, y_pred_final)

print(confusion)

[[113  11]
 [ 14  37]]


# **************************************************************** GET THE ACCURACY AND F1 IN THE VALIDATION SET ******************************************** #

In [136]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# predictions
y_pred_final = (y_prob >= 0.69).astype(int)

# confusion matrix
confusion = confusion_matrix(Y_validation_set, y_pred_final)
print(confusion)

# accuracy
accuracy = accuracy_score(Y_validation_set, y_pred_final)

# F1 score
f1 = f1_score(Y_validation_set, y_pred_final)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

[[113  11]
 [ 14  37]]
Accuracy: 0.8571428571428571
F1 Score: 0.7474747474747475


# ************************************************************** METRICS FOR VALIDATION SET ********************************************************************#
[[113  11]
 [ 14  37]]
Accuracy: 0.8571428571428571
F1 Score: 0.7474747474747475

# *****************                  OPTIMIZING THE THRESHOLD ON VALIDATION SET WITH F1 optimization → if imbalance matters               ***********************************#

In [137]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.linspace(0, 1, 100)
f1_scores = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    f1_scores.append(f1_score(Y_validation_set, y_pred))

best_idx = np.argmax(f1_scores)
best_threshold_f1 = thresholds[best_idx]

print("Best Threshold (F1):", best_threshold_f1)
print("Best F1:", f1_scores[best_idx])

Best Threshold (F1): 0.7070707070707072
Best F1: 0.7446808510638298


In [138]:
# Getting the metrics, for validation with threshold = 0.7

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# predictions
y_pred_final = (y_prob >= 0.7).astype(int)

# confusion matrix
confusion = confusion_matrix(Y_validation_set, y_pred_final)
print(confusion)

# accuracy
accuracy = accuracy_score(Y_validation_set, y_pred_final)

# F1 score
f1 = f1_score(Y_validation_set, y_pred_final)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

[[115   9]
 [ 16  35]]
Accuracy: 0.8571428571428571
F1 Score: 0.7368421052631579


# *****************                RESULTS OF THE THRESHOLD OPTIMIZATION ON VALIDATION SET WITH F1 optimization → if imbalance matters          ******************************#
[[115   9]
 [ 16  35]]
Accuracy: 0.8571428571428571
F1 Score: 0.7368421052631579

# ******************************************************************************************************************************************************************************#
# ************************************************************************   TEST DATASET            ************************************************************************   #
# ******************************************************************************************************************************************************************************#

# **********************************************                   EVALUATING THE THRESHOLD = 0.7 IN THE TEST DATASET          **************************************************#

threshold (0.7) → turns that into a decision

if ≥ 0.7 → class 1 (denial)
if < 0.7 → class 0

But the problem is that this is credit denial, if the probability is less than 0.7 then they can have the credit.

In [139]:
# build the datasets for x and y as TEST set
import pandas as pd

X_test_set = pd.DataFrame(X_part4, columns=X_bank.columns)
Y_test_set = pd.DataFrame(y_part4)
print(X_test_set.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')


In [140]:
# X_validation_set  must have only the significant variables  age, employ, address, debtinc, creddebt

print(X_test_set.columns)

X_test_set  =  X_test_set.drop(columns=['income','othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'
])
print(X_test_set.columns)

Index(['age', 'employ', 'address', 'income', 'debtinc', 'creddebt', 'othdebt',
       'ed_Did not complete high school', 'ed_High school degree',
       'ed_Post-undergraduate degree', 'ed_Some college'],
      dtype='str')
Index(['age', 'employ', 'address', 'debtinc', 'creddebt'], dtype='str')


# ****************************************************************** FITTING THE LOGISTIC RESGRESSION FOR TEST SET *************************************************** #

In [ ]:
# the datasets are:
# X_test_set 
# Y_test_set

# fitting the model with the significant variables 
# with balanced data 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix


# Add class_weight='balanced' here 👇
lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_test_set , Y_test_set)

lr_predicted = lr.predict(X_test_set)

# Confusion matrix
confusion = confusion_matrix(Y_test_set, lr_predicted)

print('Logistic regression classifier (balanced)\n', confusion)





Logistic regression classifier (balanced)
 [[95 36]
 [ 6 38]]


c:\Users\paola\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [142]:
#  Get probabilities (instead of hard predictions)
lr_predicted = lr.predict(X_test_set)
y_prob = lr.predict_proba(X_test_set)[:, 1]

# *******************************************************           Getting the confusion matrix with Test dataset and threshold = 0.7 *************************************** #

In [143]:
y_pred_final = (y_prob >= 0.7).astype(int)
confusion = confusion_matrix(Y_test_set, y_pred_final)

print(confusion)

[[118  13]
 [ 19  25]]


# ************************************************                      GET THE ACCURACY AND F1                    ************************************************ #

In [144]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# predictions
y_pred_final = (y_prob >= 0.7).astype(int)

# confusion matrix
confusion = confusion_matrix(Y_test_set, y_pred_final)
print(confusion)

# accuracy
accuracy = accuracy_score(Y_test_set, y_pred_final)

# F1 score
f1 = f1_score(Y_test_set, y_pred_final)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

[[118  13]
 [ 19  25]]
Accuracy: 0.8171428571428572
F1 Score: 0.6097560975609756


 # ************************************************                        RESULTS           ************************************************ #
Dataset: test 
thershold = 0.7
Confusion matrix:
[[118  13]
 [ 19  25]]
Accuracy: 0.817
F1 Score: 0.601


Dataset: validation
thershold = 0.7
Confusion matrix:
[[115   9]
 [ 16  35]]
Accuracy: 0.857
F1 Score: 0.737

# ******************************************************************************************************************************************************************************#
# **************************************                                    EXPLORATORY FOCUS ON THE WHOLE DATASET                      **************************************  #
# ******************************************************************************************************************************************************************************#

# ********************************     Explanatory analysis, fitting the model with the whole sample and the significant variables   ******************************** #
With threshold of 0.7

(d) For the selected model, determine how well customers are classified by obtaining appropriate metrics. First, considering that the focus of the analysis is purely explanatory (i.e., based on the full sample), so these metrics are used only as an additional measure of good or poor fit; and then from a predictive perspective, in which the goal is to evaluate the model’s performance on new data.

In [145]:
# the data is:
# X_bank, y_binary_imbalanced

# fitting the model with the significant variables 
# with balanced data 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix


# Add class_weight='balanced' here 👇
lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_bank, y_binary_imbalanced)

lr_predicted = lr.predict(X_bank)

# Confusion matrix
confusion = confusion_matrix(y_binary_imbalanced, lr_predicted)

print('Logistic regression classifier (balanced)\n', confusion)


Logistic regression classifier (balanced)
 [[393 124]
 [ 38 145]]


In [149]:
#  Get probabilities (instead of hard predictions)
lr_predicted = lr.predict(X_bank)
y_prob = lr.predict_proba(X_bank)[:, 1]

y_pred_final = (y_prob >= 0.7).astype(int)
confusion = confusion_matrix(y_binary_imbalanced, y_pred_final)

print(confusion)

[[464  53]
 [ 78 105]]


In [150]:

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# predictions
y_pred_final = (y_prob >= 0.7).astype(int)

# confusion matrix
confusion = confusion_matrix(y_binary_imbalanced, y_pred_final)
print(confusion)

# accuracy
accuracy = accuracy_score(y_binary_imbalanced, y_pred_final)

# F1 score
f1 = f1_score(y_binary_imbalanced, y_pred_final)

print("Accuracy:", accuracy)
print("F1 Score:", f1)


[[464  53]
 [ 78 105]]
Accuracy: 0.8128571428571428
F1 Score: 0.6158357771260997


 # ************************************************       COMPARISON BETWEEN TEST, VALIDATION AND THE WHOLE DATASET          ************************************************ #
Dataset: test 
thershold = 0.7
Confusion matrix:
[[118  13]
 [ 19  25]]
Accuracy: 0.817
F1 Score: 0.601


Dataset: validation
thershold = 0.7
Confusion matrix:
[[115   9]
 [ 16  35]]
Accuracy: 0.857
F1 Score: 0.737

Dataset: Whole dataset for exploration purposes (with the significant variables that were set on the training dataset)
thershold = 0.7
Confusion matrix:
[[464  53]
 [ 78 105]]
Accuracy: 0.8128571428571428
F1 Score: 0.6158357771260997

# ******************************************************************************************************************************************************************************#
# ************************************************************************      PREDICTION           ************************************************************************   #
# ******************************************************************************************************************************************************************************#

(e)	Assume there is a new client for whom we want to determine whether credit should be approved. This client is 37 years old, did not complete high school, has been in the same job for 20 years, has lived at the same address for 13 years, has an income (in thousands) of 41, a debt-to-income ratio of 12.9, credit card debt (in thousands) of 0.9, and other debts of 4.49. Would you approve the credit or not?

We the same variables the model was trained on (the statistically significant ones). 

To predict using only the significant variables, we can’t just drop columns at prediction time—you must use a model that was trained with exactly those variables.

In [151]:
# Step 1: Match the model inputs
print(X_combined.columns)


Index(['age', 'employ', 'address', 'debtinc', 'creddebt'], dtype='str')


1. Define the significant variables

In [160]:
significant_vars = ['age', 'employ', 'address', 'debtinc', 'creddebt']

2. Retrain the model using only those variables

In [161]:
X_sig = X_bank[significant_vars]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_sig, y_binary_imbalanced, random_state=0
)

lr_sig = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sig.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

3. Create the new client with the same variables

In [162]:
import pandas as pd

new_client_sig = pd.DataFrame([{
    'age': 37,
    'employ': 20,
    'address': 13,
    'debtinc': 12.9,
    'creddebt': 0.9
}])  

4. Predict probability

In [163]:
y_prob_new = lr_sig.predict_proba(new_client_sig)[:, 1]
print("Probability of denial:", y_prob_new[0])

Probability of denial: 0.01425183982196895


The model estimates only a ~1.4% chance that this client will be denied credit (class = 1)
Conversely, there is a ~98.6% chance of approval (class = 0)